In [11]:
"""
=============================================================
Análisis de Ventas — Rumbo Verde (Tienda de Productos Naturales)
=============================================================
Autor  : [Tu nombre]
Fecha  : Abril 2026
Fuentes: Bsale (tienda física) + Jumpseller (ecommerce)
Período: Abril 2026

Análisis incluidos:
  1. Rentabilidad por producto y marca (Pareto)
  2. Comparativa canal físico vs online
  3. Patrones de venta por día, hora y mapa de calor

Requisitos:
    pip install pandas matplotlib seaborn openpyxl
=============================================================
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')



In [21]:
import os
print("📍 Ubicación actual:", os.getcwd())
print("📁 Contenido:", os.listdir())

📍 Ubicación actual: /Users/cris/Documents/Ciencia de Datos/rumbo-verde-analisis/notebooks
📁 Contenido: [' 01_rentabilidad_productos.ipynb', '.ipynb_checkpoints']


In [29]:
import os
print("📍 Ubicación actual:", os.getcwd())
print("📁 Contenido carpeta data:")
print(os.listdir('../data'))

📍 Ubicación actual: /Users/cris/Documents/Ciencia de Datos/rumbo-verde-analisis/notebooks
📁 Contenido carpeta data:
['Pedidos_abril_2026.csv', '.ipynb_checkpoints', 'Detalle_de_ventas_abril_2026.xlsx']


In [30]:
# ──────────────────────────────────────────────
# CONFIGURACIÓN
# ──────────────────────────────────────────────
BSALE_FILE      = '../data/Detalle_de_ventas_abril_2026.xlsx'
JUMPSELLER_FILE = '../data/Pedidos_abril_2026.csv'
OUTPUT_DIR      = '../outputs/'
 
# Paleta de colores
VERDE  = '#2D7A4F'
AZUL   = '#2563EB'
NARANJ = '#F59E0B'
ROJO   = '#EF4444'
 
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'grid.linestyle'    : '--',
    'figure.facecolor'  : '#FAFAFA',
    'axes.facecolor'    : '#FAFAFA',
})

In [31]:
# ──────────────────────────────────────────────
# 1. CARGA Y LIMPIEZA DE DATOS
# ──────────────────────────────────────────────
 
def cargar_bsale(filepath: str) -> pd.DataFrame:
    """
    Carga el Excel de Bsale (tienda física).
    Filtra solo movimientos de venta y convierte fechas.
    """
    df = pd.read_excel(filepath)
    ventas = df[df['Tipo Movimiento'] == 'venta'].copy()
    ventas['fecha_dt'] = pd.to_datetime(ventas['Fecha Venta'], format='%d/%m/%Y')
    ventas['hora']     = ventas['Hora Venta'].astype(str).str[:2].astype(int)
    ventas['dia_semana'] = ventas['fecha_dt'].dt.day_name().map({
        'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
        'Thursday': 'Jueves', 'Friday': 'Viernes',
        'Saturday': 'Sábado', 'Sunday': 'Domingo'
    })
    print(f"  Bsale cargado: {len(ventas):,} líneas | "
          f"{ventas['Numero del documento'].nunique():,} boletas")
    return ventas
 
 
def cargar_jumpseller(filepath: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Carga el CSV de Jumpseller (ecommerce).
    Retorna:
      - df_lineas: una fila por producto por pedido
      - df_pedidos: una fila por pedido (totales agregados)
    """
    df = pd.read_csv(filepath, sep=';')
    df_pagados = df[df['Estado del Pago'] == 'Pagado'].copy()
 
    # Limpiar campos numéricos (vienen con coma decimal)
    for col in ['Total', 'Precio del Producto']:
        df_pagados[col + '_num'] = pd.to_numeric(
            df_pagados[col].astype(str).str.replace(',', '.').str.strip(),
            errors='coerce'
        )
 
    df_pedidos = df_pagados.groupby('ID').agg(
        total   = ('Total_num', 'first'),
        fecha   = ('Fecha', 'first'),
        ciudad  = ('Ciudad de Envío', 'first'),
        metodo  = ('Nombre del método de envío', 'first'),
    ).reset_index()
    df_pedidos['fecha_dt'] = pd.to_datetime(df_pedidos['fecha'].str[:10])
 
    print(f"  Jumpseller cargado: {len(df_pagados):,} líneas | "
          f"{len(df_pedidos):,} pedidos pagados")
    return df_pagados, df_pedidos
 

In [32]:
# ──────────────────────────────────────────────
# 2. ANÁLISIS 1 — RENTABILIDAD (Pareto + Marcas)
# ──────────────────────────────────────────────
 
def plot_rentabilidad(ventas: pd.DataFrame, output_path: str) -> None:
    """
    Genera figura con:
      - Top 15 productos por venta bruta (coloreados por margen)
      - Top 10 marcas: venta bruta vs % margen promedio
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle('Rumbo Verde — Análisis de Rentabilidad · Abril 2026',
                 fontsize=14, fontweight='bold', y=1.01, color='#1E293B')
 
    # — Top 15 productos —
    prod = (
        ventas.groupby('Producto / Servicio')
        .agg(venta=('Venta Total Bruta', 'sum'),
             margen=('Margen', 'sum'),
             pct_margen=('% Margen', 'mean'))
        .sort_values('venta', ascending=False)
        .head(15).reset_index()
    )
    prod['label'] = prod['Producto / Servicio'].str[:32]
    colores = [VERDE if p >= 0.35 else NARANJ if p >= 0.25 else ROJO
               for p in prod['pct_margen']]
 
    ax1 = axes[0]
    ax1.barh(prod['label'][::-1], prod['venta'][::-1] / 1e6,
             color=colores[::-1], height=0.65)
    ax1.set_xlabel('Venta Bruta (millones CLP)', fontsize=10)
    ax1.set_title('Top 15 productos por venta\n(color = nivel de rentabilidad)',
                  fontsize=11, fontweight='bold')
    ax1.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
    ax1.tick_params(axis='y', labelsize=8.5)
    leyenda = [
        mpatches.Patch(color=VERDE,  label='Margen ≥ 35%'),
        mpatches.Patch(color=NARANJ, label='Margen 25–35%'),
        mpatches.Patch(color=ROJO,   label='Margen < 25%'),
    ]
    ax1.legend(handles=leyenda, fontsize=8, loc='lower right')
 
    # — Top 10 marcas —
    marcas = (
        ventas.groupby('Marca')
        .agg(venta=('Venta Total Bruta', 'sum'),
             margen_pct=('% Margen', 'mean'))
        .sort_values('venta', ascending=False)
        .head(10).reset_index()
    )
    ax2 = axes[1]
    x = np.arange(len(marcas))
    ax2.bar(x, marcas['venta'] / 1e6, color=AZUL, alpha=0.85, label='Venta bruta')
    ax2r = ax2.twinx()
    ax2r.plot(x, marcas['margen_pct'] * 100, color=NARANJ, marker='o',
              linewidth=2, markersize=6, label='% Margen')
    ax2r.set_ylabel('% Margen promedio', fontsize=10, color=NARANJ)
    ax2r.tick_params(axis='y', colors=NARANJ)
    ax2r.spines['top'].set_visible(False)
    ax2r.set_ylim(0, 80)
    ax2.set_xticks(x)
    ax2.set_xticklabels(marcas['Marca'], rotation=35, ha='right', fontsize=9)
    ax2.set_ylabel('Venta Bruta (millones CLP)', fontsize=10)
    ax2.set_title('Top 10 marcas: Venta bruta vs % Margen',
                  fontsize=11, fontweight='bold')
    ax2.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
    lines1, lab1 = ax2.get_legend_handles_labels()
    lines2, lab2 = ax2r.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, lab1 + lab2, fontsize=8, loc='upper right')
 
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Guardado: {output_path}")

In [33]:
# ──────────────────────────────────────────────
# 3. ANÁLISIS 2 — CANAL FÍSICO VS ONLINE
# ──────────────────────────────────────────────
 
def plot_canales(ventas: pd.DataFrame, df_lineas: pd.DataFrame,
                 df_pedidos: pd.DataFrame, output_path: str) -> None:
    """
    Genera figura con:
      - Participación por canal (donut)
      - Ticket promedio y volumen de transacciones
      - Top 8 productos más vendidos online
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Rumbo Verde — Canal Físico vs Online · Abril 2026',
                 fontsize=14, fontweight='bold', y=1.01, color='#1E293B')
 
    total_bs = ventas['Venta Total Bruta'].sum()
    total_js = df_pedidos['total'].sum()
 
    # — Donut participación —
    ax = axes[0]
    wedges, texts, autotexts = ax.pie(
        [total_bs / 1e6, total_js / 1e6],
        labels=['Tienda física\n(Bsale)', 'Online\n(Jumpseller)'],
        colors=[AZUL, VERDE],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(width=0.6), textprops=dict(fontsize=10)
    )
    for at in autotexts:
        at.set_fontsize(11); at.set_fontweight('bold'); at.set_color('white')
    ax.set_title(f'Participación en ventas totales\nTotal: '
                 f'${(total_bs + total_js) / 1e6:.1f}M CLP',
                 fontsize=11, fontweight='bold')
 
    # — Ticket promedio y volumen —
    ax = axes[1]
    ticket_bs = ventas.groupby('Numero del documento')['Venta Total Bruta'].sum().mean()
    ticket_js = df_pedidos['total'].mean()
    n_bs = ventas['Numero del documento'].nunique()
    n_js = len(df_pedidos)
    categorias = ['Ticket promedio\n(miles CLP)', 'N° transacciones']
    x = np.arange(len(categorias))
    w = 0.35
    b1 = ax.bar(x - w/2, [ticket_bs/1000, n_bs], w,
                label='Tienda física', color=AZUL, alpha=0.85)
    b2 = ax.bar(x + w/2, [ticket_js/1000, n_js], w,
                label='Online', color=VERDE, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(categorias, fontsize=10)
    ax.set_title('Ticket promedio vs Volumen de transacciones',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 1,
                    f'{h:.0f}', ha='center', va='bottom', fontsize=9)
 
    # — Top 8 productos online —
    ax = axes[2]
    top_online = (
        df_lineas.groupby('Nombre del Producto')['Cantidad de Productos']
        .apply(lambda x: pd.to_numeric(
            x.astype(str).str.replace(',', '.'), errors='coerce').sum())
        .sort_values(ascending=False)
        .head(8)
    )
    top_online.index = top_online.index.str[:33]
    ax.barh(range(len(top_online)), top_online.values[::-1],
            color=VERDE, alpha=0.85)
    ax.set_yticks(range(len(top_online)))
    ax.set_yticklabels(top_online.index[::-1], fontsize=8.5)
    ax.set_xlabel('Unidades vendidas', fontsize=10)
    ax.set_title('Top 8 productos más vendidos\nonline (Jumpseller)',
                 fontsize=11, fontweight='bold')
    for i, v in enumerate(top_online.values[::-1]):
        ax.text(v + 0.3, i, str(int(v)), va='center', fontsize=9)
 
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Guardado: {output_path}")

In [34]:
# ──────────────────────────────────────────────
# 4. ANÁLISIS 3 — PATRONES DE VENTA
# ──────────────────────────────────────────────
 
def plot_patrones(ventas: pd.DataFrame, df_pedidos: pd.DataFrame,
                  output_path: str) -> None:
    """
    Genera figura 2×2 con:
      - Ventas diarias (tienda + online superpuestos)
      - Ventas por día de semana
      - Ventas por franja horaria
      - Mapa de calor día × hora
    """
    dias_order = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle('Rumbo Verde — Patrones de Venta · Abril 2026',
                 fontsize=14, fontweight='bold', y=1.01, color='#1E293B')
 
    # — Ventas diarias —
    ax = axes[0, 0]
    diario_bs = ventas.groupby('fecha_dt')['Venta Total Bruta'].sum()
    diario_js = df_pedidos.groupby('fecha_dt')['total'].sum()
    ax.fill_between(diario_bs.index, diario_bs / 1e6, alpha=0.2, color=AZUL)
    ax.plot(diario_bs.index, diario_bs / 1e6, color=AZUL, linewidth=2,
            label='Tienda física')
    if len(diario_js) > 2:
        ax.fill_between(diario_js.index, diario_js / 1e6, alpha=0.2, color=VERDE)
        ax.plot(diario_js.index, diario_js / 1e6, color=VERDE, linewidth=2,
                linestyle='--', label='Online')
    ax.set_title('Ventas diarias — Abril 2026', fontsize=11, fontweight='bold')
    ax.set_ylabel('Millones CLP', fontsize=10)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=30)
 
    # — Por día de semana —
    ax = axes[0, 1]
    por_dia = ventas.groupby('dia_semana')['Venta Total Bruta'].sum().reindex(dias_order)
    colors_dia = [VERDE if d in ['Miércoles', 'Jueves'] else AZUL for d in dias_order]
    ax.bar(dias_order, por_dia / 1e6, color=colors_dia, alpha=0.85)
    ax.set_title('Ventas por día de semana', fontsize=11, fontweight='bold')
    ax.set_ylabel('Millones CLP', fontsize=10)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
    ax.tick_params(axis='x', rotation=25)
    for i, v in enumerate(por_dia):
        ax.text(i, v/1e6 + 0.05, f'${v/1e6:.1f}M', ha='center', fontsize=8.5)
 
    # — Por hora —
    ax = axes[1, 0]
    por_hora = ventas.groupby('hora')['Venta Total Bruta'].sum().sort_index()
    colors_hora = [VERDE if h in [11, 12] else AZUL for h in por_hora.index]
    ax.bar(por_hora.index, por_hora / 1e6, color=colors_hora, alpha=0.85)
    ax.set_title('Ventas por franja horaria (tienda física)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Hora del día', fontsize=10)
    ax.set_ylabel('Millones CLP', fontsize=10)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
    ax.set_xticks(por_hora.index)
    ax.set_xticklabels([f'{h}:00' for h in por_hora.index], rotation=35, fontsize=8.5)
 
    # — Heatmap día × hora —
    ax = axes[1, 1]
    ventas['dia_num'] = ventas['fecha_dt'].dt.dayofweek
    pivot = ventas.pivot_table(
        values='Venta Total Bruta', index='dia_num',
        columns='hora', aggfunc='sum', fill_value=0
    )
    pivot.index = [dias_order[i] for i in pivot.index]
    sns.heatmap(pivot / 1e6, ax=ax, cmap='YlGn', linewidths=0.3,
                fmt='.1f', annot=True, annot_kws={'size': 7.5},
                cbar_kws={'label': 'Millones CLP'})
    ax.set_title('Mapa de calor: Día × Hora', fontsize=11, fontweight='bold')
    ax.set_xlabel('Hora', fontsize=10)
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=35)
 
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Guardado: {output_path}")
 

In [35]:
# ──────────────────────────────────────────────
# 5. RESUMEN EN CONSOLA
# ──────────────────────────────────────────────
 
def imprimir_resumen(ventas: pd.DataFrame, df_pedidos: pd.DataFrame) -> None:
    total_bs = ventas['Venta Total Bruta'].sum()
    total_js = df_pedidos['total'].sum()
    margen_bs = ventas['% Margen'].mean() * 100
    ticket_js = df_pedidos['total'].mean()
    n_bs = ventas['Numero del documento'].nunique()
    n_js = len(df_pedidos)
 
    print("\n" + "="*55)
    print("  RESUMEN EJECUTIVO — RUMBO VERDE · ABRIL 2026")
    print("="*55)
    print(f"  Ventas tienda física (Bsale):  ${total_bs/1e6:.2f}M CLP")
    print(f"  Ventas online (Jumpseller):     ${total_js/1e6:.2f}M CLP")
    print(f"  Venta total estimada:           ${(total_bs+total_js)/1e6:.2f}M CLP")
    print(f"  Canal online / total:           {total_js/(total_bs+total_js)*100:.1f}%")
    print(f"  Margen promedio tienda:         {margen_bs:.1f}%")
    print(f"  Boletas tienda:                 {n_bs:,}")
    print(f"  Pedidos online pagados:         {n_js:,}")
    print(f"  Ticket promedio online:         ${ticket_js:,.0f} CLP")
    print("="*55)
 

In [36]:
# ──────────────────────────────────────────────
# MAIN — Ejecutar todo el análisis
# ──────────────────────────────────────────────

print("\n📦 Cargando datos...")
ventas                = cargar_bsale(BSALE_FILE)
df_lineas, df_pedidos = cargar_jumpseller(JUMPSELLER_FILE)

imprimir_resumen(ventas, df_pedidos)

print("\n📊 Generando análisis...")
plot_rentabilidad(ventas, OUTPUT_DIR + 'rv_01_rentabilidad.png')
plot_canales(ventas, df_lineas, df_pedidos, OUTPUT_DIR + 'rv_02_canales.png')
plot_patrones(ventas, df_pedidos, OUTPUT_DIR + 'rv_03_patrones.png')

print("\n✅ Análisis completo. Imágenes guardadas en:", OUTPUT_DIR)


📦 Cargando datos...
  Bsale cargado: 2,399 líneas | 1,060 boletas
  Jumpseller cargado: 436 líneas | 436 pedidos pagados

  RESUMEN EJECUTIVO — RUMBO VERDE · ABRIL 2026
  Ventas tienda física (Bsale):  $33.70M CLP
  Ventas online (Jumpseller):     $16.61M CLP
  Venta total estimada:           $50.31M CLP
  Canal online / total:           33.0%
  Margen promedio tienda:         41.6%
  Boletas tienda:                 1,060
  Pedidos online pagados:         436
  Ticket promedio online:         $38,100 CLP

📊 Generando análisis...
  Guardado: ../outputs/rv_01_rentabilidad.png
  Guardado: ../outputs/rv_02_canales.png
  Guardado: ../outputs/rv_03_patrones.png

✅ Análisis completo. Imágenes guardadas en: ../outputs/


In [37]:
# Verificar si las ventas online de Jumpseller están dentro de Bsale
print("=== BSALE ===")
print(ventas['Tipo de entrega'].value_counts())
print()
print("=== BSALE — Tipos de documento ===")
print(ventas['Tipo de Documento'].value_counts())
print()
print("=== BSALE — Lista de precio (puede indicar canal) ===")
print(ventas['Lista de Precio'].value_counts())

=== BSALE ===
Tipo de entrega
Entrega inmediata    2399
Name: count, dtype: int64

=== BSALE — Tipos de documento ===
Tipo de Documento
BOLETA ELECTRÓNICA T                       2370
FACTURA ELECTRÓNICA                          28
BOLETA NO AFECTA O EXENTA ELECTRÓNICA T       1
Name: count, dtype: int64

=== BSALE — Lista de precio (puede indicar canal) ===
Lista de Precio
Lista de precios base RV Final    2003
Uber 2026 - Plantas y Tienda       316
Uber Rappi - Septiembre             48
UBER 2026                           25
Uber Plantas - Abril                 5
ML FULL - No usar                    2
Name: count, dtype: int64


In [38]:
# Desglose correcto por canal
uber_rappi = ventas[ventas['Lista de Precio'].str.contains('Uber|Rappi|ML', case=False, na=False)]
tienda_fisica = ventas[~ventas['Lista de Precio'].str.contains('Uber|Rappi|ML', case=False, na=False)]

print("=== DESGLOSE REAL POR CANAL ===")
print(f"Tienda física pura:      ${tienda_fisica['Venta Total Bruta'].sum()/1e6:.2f}M CLP ({len(tienda_fisica['Numero del documento'].unique()):,} boletas)")
print(f"Uber/Rappi/Delivery:     ${uber_rappi['Venta Total Bruta'].sum()/1e6:.2f}M CLP ({len(uber_rappi['Numero del documento'].unique()):,} boletas)")
print(f"Ecommerce (Jumpseller):  ${df_pedidos['total'].sum()/1e6:.2f}M CLP ({len(df_pedidos):,} pedidos)")
print(f"─────────────────────────────────────")
total = tienda_fisica['Venta Total Bruta'].sum() + uber_rappi['Venta Total Bruta'].sum() + df_pedidos['total'].sum()
print(f"TOTAL REAL:              ${total/1e6:.2f}M CLP")

=== DESGLOSE REAL POR CANAL ===
Tienda física pura:      $25.56M CLP (805 boletas)
Uber/Rappi/Delivery:     $8.14M CLP (255 boletas)
Ecommerce (Jumpseller):  $16.61M CLP (436 pedidos)
─────────────────────────────────────
TOTAL REAL:              $50.31M CLP


In [39]:
# Verificar qué está inflando el número
print("=== TIPOS DE MOVIMIENTO en el Excel ===")
import pandas as pd
df_raw = pd.read_excel('../data/Detalle_de_ventas_abril_2026.xlsx')
print(df_raw['Tipo Movimiento'].value_counts())
print()
print("=== SUMA POR TIPO DE MOVIMIENTO ===")
print(df_raw.groupby('Tipo Movimiento')['Venta Total Bruta'].sum())
print()
print("=== TIPOS DE DOCUMENTO ===")
print(df_raw.groupby('Tipo de Documento')['Venta Total Bruta'].sum())

=== TIPOS DE MOVIMIENTO en el Excel ===
Tipo Movimiento
venta         2399
devolucion      13
Name: count, dtype: int64

=== SUMA POR TIPO DE MOVIMIENTO ===
Tipo Movimiento
devolucion     -121263.0
venta         33697376.0
Name: Venta Total Bruta, dtype: float64

=== TIPOS DE DOCUMENTO ===
Tipo de Documento
BOLETA ELECTRÓNICA T                       33208783.0
BOLETA NO AFECTA O EXENTA ELECTRÓNICA T       52650.0
FACTURA ELECTRÓNICA                          435943.0
NOTA DE CRÉDITO ELECTRÓNICA                 -121263.0
Name: Venta Total Bruta, dtype: float64


In [40]:
print("=== COMPARACIÓN NETO vs BRUTO ===")
ventas_netas  = df_raw[df_raw['Tipo Movimiento']=='venta']['Venta Total Neta'].sum()
ventas_brutas = df_raw[df_raw['Tipo Movimiento']=='venta']['Venta Total Bruta'].sum()
devoluciones  = df_raw[df_raw['Tipo Movimiento']=='devolucion']['Venta Total Bruta'].sum()
impuestos     = df_raw[df_raw['Tipo Movimiento']=='venta']['Total Impuestos'].sum()

print(f"Venta Total Bruta:     ${ventas_brutas/1e6:.3f}M")
print(f"Devoluciones:          ${devoluciones/1e6:.3f}M")
print(f"Total Impuestos (IVA): ${impuestos/1e6:.3f}M")
print(f"Venta Total Neta:      ${ventas_netas/1e6:.3f}M")
print(f"Bruta - IVA + devol:   ${(ventas_brutas + devoluciones - impuestos)/1e6:.3f}M")
print(f"Bsale dashboard:       $28.224M")


=== COMPARACIÓN NETO vs BRUTO ===
Venta Total Bruta:     $33.697M
Devoluciones:          $-0.121M
Total Impuestos (IVA): $5.372M
Venta Total Neta:      $28.326M
Bruta - IVA + devol:   $28.204M
Bsale dashboard:       $28.224M
